# Day 7 — Probability basics, conditional probability and Bayes intuition

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5 and 6, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column). Probabilities here describe picking a student at random from this file.

## 1. Introduction to probability

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-007/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
from fractions import Fraction

# A fair die: the sample space is every possible outcome, each equally likely
die = [1, 2, 3, 4, 5, 6]
three = [x for x in die if x == 3]                 # the event "roll a 3"
even = [x for x in die if x % 2 == 0]              # the event "roll an even number"
print("Sample space:", die)
print("P(3) =", Fraction(len(three), len(die)), "=", round(len(three) / len(die), 3))
print("P(even) =", Fraction(len(even), len(die)), "=", len(even) / len(die))

Sample space: [1, 2, 3, 4, 5, 6]
P(3) = 1/6 = 0.167
P(even) = 1/2 = 0.5


In [ ]:
import pandas as pd

# The same 150 students as Days 5 and 6: 8 real ones and 142 synthetic ones
df = pd.read_csv("student_data_150.csv")

# Probability that a randomly selected student is in class A
total_students = len(df)
class_a = (df["Class"] == "A").sum()
probability = class_a / total_students
print("Total students:", total_students)
print("Class A students:", class_a)
print("Probability:", round(probability, 3))

Total students: 150
Class A students: 49
Probability: 0.327


In [ ]:
# More events: count the students in the event, divide by all students
events = {
    "Class B": df["Class"] == "B",
    "ExamScore above 80": df["ExamScore"] > 80,
    "Attendance above 90%": df["Attendance"] > 90,
    "StudyHours above 6": df["StudyHours"] > 6,
    "Passed": df["Passed"],
}
for name, event in events.items():
    print(f"P({name}) = {event.sum()}/{len(df)} = {event.mean():.3f}")

P(Class B) = 49/150 = 0.327
P(ExamScore above 80) = 8/150 = 0.053
P(Attendance above 90%) = 22/150 = 0.147
P(StudyHours above 6) = 2/150 = 0.013
P(Passed) = 106/150 = 0.707


In [ ]:
import numpy as np

# Probability as a long-run frequency: roll a fair die many times and count the 3s
rng = np.random.default_rng(7)                  # a fixed seed, so everyone gets the same rolls
for n in [10, 100, 1000, 100000]:
    rolls = rng.integers(1, 7, size=n)
    print(f"{n:>6} rolls: share of 3s = {(rolls == 3).mean():.3f}   (exact: {1 / 6:.3f})")

    10 rolls: share of 3s = 0.000   (exact: 0.167)
   100 rolls: share of 3s = 0.170   (exact: 0.167)
  1000 rolls: share of 3s = 0.172   (exact: 0.167)
100000 rolls: share of 3s = 0.166   (exact: 0.167)


## 2. Sample space and events

In [ ]:
# Sample space and an event as Python sets
sample_space = {1, 2, 3, 4, 5, 6}
event_a = {2, 4, 6}                                  # even numbers
probability = len(event_a) / len(sample_space)
print("Sample space:", sample_space)
print("Event A:", event_a)
print("Probability:", probability)

Sample space: {1, 2, 3, 4, 5, 6}
Event A: {2, 4, 6}
Probability: 0.5


In [ ]:
from itertools import product

# Tossing two coins: every combination of the two results
coins = list(product(["H", "T"], repeat=2))
at_least_one_head = [c for c in coins if "H" in c]
print("Sample space:", coins)
print("At least one head:", at_least_one_head, "-> P =", len(at_least_one_head) / len(coins))

Sample space: [('H', 'H'), ('H', 'T'), ('T', 'H'), ('T', 'T')]
At least one head: [('H', 'H'), ('H', 'T'), ('T', 'H')] -> P = 0.75


In [ ]:
# Combining events with sets: A = even, B = greater than 4
A = {2, 4, 6}
B = {5, 6}
S = sample_space
print("A or B (union):", A | B, "-> P =", round(len(A | B) / len(S), 3))
print("A and B (intersection):", A & B, "-> P =", round(len(A & B) / len(S), 3))
print("not A (complement):", S - A, "-> P =", len(S - A) / len(S))

A or B (union): {2, 4, 5, 6} -> P = 0.667
A and B (intersection): {6} -> P = 0.167
not A (complement): {1, 3, 5} -> P = 0.5


In [ ]:
import pandas as pd

df = pd.read_csv("student_data_150.csv")             # 150 students: 8 real, 142 synthetic
n = len(df)
A = df["Class"] == "A"                               # event A: in class A
B = df["ExamScore"] > 80                             # event B: exam score above 80
print(f"P(A) = {A.sum()}/{n} = {A.mean():.3f}")
print(f"P(B) = {B.sum()}/{n} = {B.mean():.3f}")
print(f"P(A and B) = {(A & B).sum()}/{n} = {(A & B).mean():.3f}")
print(f"P(A or B) = {(A | B).sum()}/{n} = {(A | B).mean():.3f}")
print()
print(pd.crosstab(A.rename("Class A"), B.rename("Score > 80"), margins=True, margins_name="Total"))

P(A) = 49/150 = 0.327
P(B) = 8/150 = 0.053
P(A and B) = 3/150 = 0.020
P(A or B) = 54/150 = 0.360

Score > 80  False  True  Total
Class A                       
False          96     5    101
True           46     3     49
Total         142     8    150


In [ ]:
# More events from the practice challenges
print(f"P(class C) = {(df['Class'] == 'C').mean():.3f}")
print(f"P(attendance below 75%) = {(df['Attendance'] < 75).sum()}/{n} = {(df['Attendance'] < 75).mean():.3f}")

P(class C) = 0.313
P(attendance below 75%) = 49/150 = 0.327


In [ ]:
from fractions import Fraction

# Picking two students WITHOUT replacement makes the second pick depend on the first
passed = int(df["Passed"].sum())
print("Students who passed:", passed, "of", n)
print("P(first passed) =", Fraction(passed, n), "=", round(passed / n, 4))
print("P(second passed | first passed) =", Fraction(passed - 1, n - 1), "=", round((passed - 1) / (n - 1), 4))
print("P(second passed | first failed) =", Fraction(passed, n - 1), "=", round(passed / (n - 1), 4))

Students who passed: 106 of 150
P(first passed) = 53/75 = 0.7067
P(second passed | first passed) = 105/149 = 0.7047
P(second passed | first failed) = 106/149 = 0.7114


## 3. Classical probability

In [ ]:
# Classical probability: favourable outcomes / all outcomes, when every outcome is equally likely
sample_space = [1, 2, 3, 4, 5, 6]
favourable = [x for x in sample_space if x > 4]
probability = len(favourable) / len(sample_space)
print("Favourable outcomes:", favourable)
print("Probability:", round(probability, 3))

Favourable outcomes: [5, 6]
Probability: 0.333


In [ ]:
# Three fair coins: P(exactly two heads)
tosses = list(product("HT", repeat=3))
two_heads = [t for t in tosses if t.count("H") == 2]
print(len(tosses), "outcomes; exactly two heads:", ["".join(t) for t in two_heads])
print("P(exactly two heads) =", Fraction(len(two_heads), len(tosses)))

8 outcomes; exactly two heads: ['HHT', 'HTH', 'THH']
P(exactly two heads) = 3/8


In [ ]:
# A standard deck: 13 ranks x 4 suits = 52 equally likely cards
ranks = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
suits = ["hearts", "diamonds", "clubs", "spades"]
deck = list(product(ranks, suits))
red = [c for c in deck if c[1] in ("hearts", "diamonds")]
spades = [c for c in deck if c[1] == "spades"]
aces = [c for c in deck if c[0] == "A"]
print("Cards:", len(deck))
print("P(red) =", Fraction(len(red), len(deck)), "| P(spade) =", Fraction(len(spades), len(deck)), "| P(ace) =", Fraction(len(aces), len(deck)))

Cards: 52
P(red) = 1/2 | P(spade) = 1/4 | P(ace) = 1/13


In [ ]:
# The limitation: outcomes that are NOT equally likely. The sum of two dice can be 2 to 12...
sums = [a + b for a, b in product(range(1, 7), repeat=2)]
for total in [2, 7, 12]:
    print(f"P(sum = {total:>2}) = {Fraction(sums.count(total), len(sums))}   (not 1/11)")

P(sum =  2) = 1/36   (not 1/11)
P(sum =  7) = 1/6   (not 1/11)
P(sum = 12) = 1/36   (not 1/11)


In [ ]:
# Empirical probability: a share observed in data, an estimate rather than an exact value
probability = (df["ExamScore"] > 80).mean()
print("Probability of score above 80:", round(probability, 3))
real = df[df["Source"] == "original"]
print("In the 8 real students only:", round((real["ExamScore"] > 80).mean(), 3))
print("Classes in the data:", sorted(df["Class"].unique()), "| P(Unknown) =", round((df["Class"] == "Unknown").mean(), 3))

Probability of score above 80: 0.053
In the 8 real students only: 0.0
Classes in the data: ['A', 'B', 'C', 'Unknown'] | P(Unknown) = 0.033


## 4. Conditional probability

In [ ]:
# Conditional probability on a die: P(6 | even) = P(6 and even) / P(even)
even = {2, 4, 6}
six_and_even = {6} & even
print("P(6 | even) =", Fraction(len(six_and_even), len(even)), "| P(6) =", Fraction(1, 6))

P(6 | even) = 1/3 | P(6) = 1/6


In [ ]:
# P(score > 80 | attendance > 90): count A and B, divide by the count of B (check B isn't empty first)
attendance_high = df["Attendance"] > 90
score_high = df["ExamScore"] > 80
both = attendance_high & score_high
if attendance_high.sum() > 0:
    probability = both.sum() / attendance_high.sum()
    print("Students with attendance > 90:", attendance_high.sum())
    print("Of those, score > 80:", both.sum())
    print("P(Score > 80 | Attendance > 90):", round(probability, 3))
    print("P(Score > 80) for everyone:", round(score_high.mean(), 3))

Students with attendance > 90: 22
Of those, score > 80: 1
P(Score > 80 | Attendance > 90): 0.045
P(Score > 80) for everyone: 0.053


In [ ]:
# A clearer case with more students: does knowing study hours change the chance of passing?
def conditional(a, b):
    # P(a | b) = P(a and b) / P(b); undefined when b never happens
    if b.sum() == 0:
        raise ValueError("P(B) is 0: the condition never happens")
    return (a & b).sum() / b.sum()

study_high = df["StudyHours"] >= 4.5
passed = df["Passed"]
print("Students who study 4.5 hours or more:", study_high.sum())
print("P(Passed) =", round(passed.mean(), 3))
print("P(Passed | StudyHours >= 4.5) =", round(conditional(passed, study_high), 3))
print("P(Passed | StudyHours < 4.5) =", round(conditional(passed, ~study_high), 3))

Students who study 4.5 hours or more: 44
P(Passed) = 0.707
P(Passed | StudyHours >= 4.5) = 0.932
P(Passed | StudyHours < 4.5) = 0.613


In [ ]:
# P(attendance > 90 | class) for each class: the condition is a group
for cls in ["A", "B", "C"]:
    in_class = df["Class"] == cls
    print(f"P(Attendance > 90 | class {cls}) = {(attendance_high & in_class).sum()}/{in_class.sum()} = {conditional(attendance_high, in_class):.3f}")

P(Attendance > 90 | class A) = 7/49 = 0.143
P(Attendance > 90 | class B) = 10/49 = 0.204
P(Attendance > 90 | class C) = 4/47 = 0.085


In [ ]:
# Independence check: A and B are independent if P(A | B) = P(A), or equivalently P(A and B) = P(A) x P(B)
class_a = df["Class"] == "A"
print(f"P(Passed) = {passed.mean():.3f} | P(Passed | class A) = {conditional(passed, class_a):.3f}")
print(f"P(Passed) x P(class A) = {passed.mean() * class_a.mean():.3f} | P(Passed and class A) = {(passed & class_a).mean():.3f}")

P(Passed) = 0.707 | P(Passed | class A) = 0.714
P(Passed) x P(class A) = 0.231 | P(Passed and class A) = 0.233


## 5. Bayes intuition

In [ ]:
# Medical testing with Bayes' theorem
prior = 0.01            # P(disease): 1% of people have it (the base rate)
sensitivity = 0.99      # P(positive | disease): the test finds 99% of real cases
specificity = 0.95      # P(negative | no disease): 95% of healthy people test negative
false_positive = 1 - specificity      # P(positive | no disease) = 0.05

evidence = sensitivity * prior + false_positive * (1 - prior)      # P(positive), from both groups
posterior = sensitivity * prior / evidence                          # P(disease | positive)
print("P(positive):", round(evidence, 4))
print("Posterior probability:", round(posterior, 3))
print("Percentage:", round(posterior * 100, 2), "%")

P(positive): 0.0594
Posterior probability: 0.167
Percentage: 16.67 %


In [ ]:
# The same answer with counts: imagine 10,000 people
people = 10_000
sick = people * prior                              # 100 people have the disease
healthy = people - sick                            # 9,900 don't
true_pos = sick * sensitivity                      # 99 sick people test positive
false_pos = healthy * false_positive               # 495 healthy people test positive too
print(f"Positive tests: {true_pos:.0f} sick + {false_pos:.0f} healthy = {true_pos + false_pos:.0f}")
print(f"P(disease | positive) = {true_pos:.0f} / {true_pos + false_pos:.0f} = {true_pos / (true_pos + false_pos):.3f}")

Positive tests: 99 sick + 495 healthy = 594
P(disease | positive) = 99 / 594 = 0.167


In [ ]:
# Spam detection: 20% of emails are spam; "offer" appears in 60% of spam and 5% of other emails
p_spam = 0.20
p_offer_given_spam = 0.60
p_offer_given_ham = 0.05
p_offer = p_offer_given_spam * p_spam + p_offer_given_ham * (1 - p_spam)
print("P(offer):", round(p_offer, 2))
print("P(spam | offer):", round(p_offer_given_spam * p_spam / p_offer, 3))

P(offer): 0.16
P(spam | offer): 0.75


In [ ]:
# Base rates matter: the same test, for diseases of different prevalence
for base_rate in [0.001, 0.01, 0.10, 0.50]:
    p_pos = sensitivity * base_rate + false_positive * (1 - base_rate)
    print(f"Prevalence {base_rate:>5.1%}: P(disease | positive) = {sensitivity * base_rate / p_pos:.3f}")

Prevalence  0.1%: P(disease | positive) = 0.019
Prevalence  1.0%: P(disease | positive) = 0.167
Prevalence 10.0%: P(disease | positive) = 0.687
Prevalence 50.0%: P(disease | positive) = 0.952


In [ ]:
# Updating again: a second, independent positive test starts from the first posterior
belief = prior
for test in [1, 2, 3]:
    p_pos = sensitivity * belief + false_positive * (1 - belief)
    belief = sensitivity * belief / p_pos
    print(f"After positive test {test}: P(disease) = {belief:.3f}")

After positive test 1: P(disease) = 0.167
After positive test 2: P(disease) = 0.798
After positive test 3: P(disease) = 0.987


In [ ]:
# On the student data: Bayes' theorem gives the same answer as counting directly
A = df["Passed"]                       # the event we care about
B = df["StudyHours"] >= 4.5            # the evidence we observe
prior = A.mean()                       # P(A)
likelihood = conditional(B, A)         # P(B | A)
evidence = B.mean()                    # P(B)
print(f"Prior P(Passed) = {prior:.3f}")
print(f"Likelihood P(StudyHours >= 4.5 | Passed) = {likelihood:.3f}")
print(f"Evidence P(StudyHours >= 4.5) = {evidence:.3f}")
print(f"Posterior by Bayes = {likelihood * prior / evidence:.3f} | counted directly = {conditional(A, B):.3f}")

Prior P(Passed) = 0.707
Likelihood P(StudyHours >= 4.5 | Passed) = 0.387
Evidence P(StudyHours >= 4.5) = 0.293
Posterior by Bayes = 0.932 | counted directly = 0.932


## 6. Mini project: probability analysis of student performance

In [ ]:
# Mini project: load the dataset (empirical probabilities: shares of these 150 students)
df = pd.read_csv("student_data_150.csv")
print("Students:", len(df), "|", df["Source"].value_counts().to_dict())

def p_given(a, b):
    # P(a | b), or None when the condition b never happens (a zero denominator)
    return None if b.sum() == 0 else (a & b).sum() / b.sum()

score_high = df["ExamScore"] > 80
attendance_high = df["Attendance"] > 90
print("P(Score > 80):", round(score_high.mean(), 3))
print("P(Attendance > 90):", round(attendance_high.mean(), 3))
print("P(Score > 80 and Attendance > 90):", round((score_high & attendance_high).mean(), 3))
print("P(Score > 80 | Attendance > 90):", round(p_given(score_high, attendance_high), 3))
print("P(Attendance > 90 | Score > 80):", round(p_given(attendance_high, score_high), 3))

Students: 150 | {'synthetic': 142, 'original': 8}
P(Score > 80): 0.053
P(Attendance > 90): 0.147
P(Score > 80 and Attendance > 90): 0.007
P(Score > 80 | Attendance > 90): 0.045
P(Attendance > 90 | Score > 80): 0.125


In [ ]:
# Probability table: each class, and a high score within each class
rows = []
for cls in ["A", "B", "C", "Unknown", "Class 5"]:
    in_class = df["Class"] == cls
    p_score = p_given(score_high, in_class)
    rows.append({"class": cls, "students": int(in_class.sum()), "P(class)": round(in_class.mean(), 3),
                 "P(Score > 80 | class)": "undefined" if p_score is None else round(p_score, 3)})
print(pd.DataFrame(rows).to_string(index=False))

  class  students  P(class) P(Score > 80 | class)
      A        49     0.327                 0.061
      B        49     0.327                 0.061
      C        47     0.313                 0.043
Unknown         5     0.033                   0.0
Class 5         0     0.000             undefined


In [ ]:
# Independence: compare P(A | B) with P(A), and P(A and B) with P(A) x P(B)
passed = df["Passed"]
attendance_85 = df["Attendance"] >= 85
print(f"P(Passed) = {passed.mean():.3f} | P(Passed | Attendance >= 85) = {p_given(passed, attendance_85):.3f} ({attendance_85.sum()} students)")
print(f"P(Score > 80) = {score_high.mean():.3f} | P(Score > 80 | Attendance > 90) = {p_given(score_high, attendance_high):.3f} ({(score_high & attendance_high).sum()} student in both)")
print(f"P(Score > 80) x P(Attendance > 90) = {score_high.mean() * attendance_high.mean():.4f} | P(both) = {(score_high & attendance_high).mean():.4f}")

P(Passed) = 0.707 | P(Passed | Attendance >= 85) = 0.891 (46 students)
P(Score > 80) = 0.053 | P(Score > 80 | Attendance > 90) = 0.045 (1 student in both)
P(Score > 80) x P(Attendance > 90) = 0.0078 | P(both) = 0.0067


In [ ]:
# Theoretical: the medical-testing example, kept separate from the student data
def bayes(prior, sensitivity, specificity):
    false_positive = 1 - specificity
    evidence = sensitivity * prior + false_positive * (1 - prior)
    return sensitivity * prior / evidence

print("P(disease | positive), 1% prevalence:", round(bayes(0.01, 0.99, 0.95), 3))
print("P(disease | positive), 5% prevalence:", round(bayes(0.05, 0.99, 0.95), 3))

P(disease | positive), 1% prevalence: 0.167
P(disease | positive), 5% prevalence: 0.51
